### To download the report about gilts, go here: https://www.dmo.gov.uk/data/pdfdatareport?reportCode=D1A

In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import pandas as pd
from datetime import datetime

### Load current gilts prices

In [4]:
# -----------------------------
# Step 1: Load the HL gilts page
# -----------------------------
url = "https://www.hl.co.uk/shares/corporate-bonds-gilts/bond-prices/uk-gilts?column=maturity&order=asc"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}
response = requests.get(url, headers=headers)
if response.status_code != 200:
    raise Exception(f"Failed to load page, status code: {response.status_code}")

html = response.text

# -----------------------------
# Step 2: Parse HTML
# -----------------------------
soup = BeautifulSoup(html, "html.parser")

# Find all rows containing gilts
rows = soup.find_all("tr", class_=["table-alt", "table-row"])  # both normal and alt rows

data = []

# -----------------------------
# Step 3: Extract fields
# -----------------------------
for row in rows:
    cells = row.find_all("td")
    if len(cells) < 5:
        continue  # skip malformed rows

    # Issuer from <a class="link-headline">
    issuer_tag = cells[0].find("a", class_="link-headline")
    issuer = issuer_tag.get_text(strip=True) if issuer_tag else ""

    # Coupon (%) from second column
    try:
        coupon = float(cells[1].get_text(strip=True).replace(",", ""))
    except:
        coupon = None

    # Maturity from third column
    maturity = cells[2].get_text(strip=True)

    # Price from fourth column
    try:
        price = float(cells[3].get_text(strip=True).replace(",", ""))
    except:
        price = None

    # Actions from fifth column (text of all <a> tags)
    actions = ", ".join([a.get_text(strip=True) for a in cells[4].find_all("a")])

    data.append({
        "Issuer": issuer,
        "Coupon (%)": coupon,
        "Maturity": maturity,
        "Price": price,
    })

# -----------------------------
# Step 4: Create DataFrame
# -----------------------------
gilts_df = pd.DataFrame(data)

# Optional: parse Maturity to datetime
gilts_df["Maturity"] = pd.to_datetime(gilts_df["Maturity"], errors="coerce")

# Show results
print(f"Total gilts extracted: {len(gilts_df)}")
gilts_df.head()

Total gilts extracted: 36


,Issuer,Coupon (%),Maturity,Price
0,Treasury 0.375% 22/10/2026,0.375,2026-10-22,98.04
1,Treasury 3.75% 07/03/2027,3.750,2027-03-07,99.44
2,Treasury 4.25% 07/12/2027,4.250,2027-12-07,99.82
3,Treasury 0.125% 31/01/2028,0.125,2028-01-31,92.80
4,Treasury 4.5% 07/06/2028,4.500,2028-06-07,100.14


### Load gilt information

In [5]:
gilts_info = pd.read_excel("gilts_info.xls")

In [6]:
# Step 1: set row 7 as header
gilts_info.columns = gilts_info.iloc[7]

# Step 2: drop all rows above row 7 (including row 7 itself)
gilts_info = gilts_info.iloc[8:].reset_index(drop=True)

# Step 3: keep only columns by position: 0,2,3,4,5
cols_to_keep = [0, 1, 2, 3, 4, 5]
gilts_info = gilts_info.iloc[:, cols_to_keep].copy()

# Optional: clean column names (strip whitespace and newlines)
gilts_info.columns = gilts_info.columns.str.replace("\n", " ").str.strip()

# Clean column names
gilts_info.columns = (
    gilts_info.columns
    .str.replace("\n", " ")   # replace newlines with space
    .str.replace(r"\s+", " ", regex=True)  # collapse multiple spaces
    .str.strip()  # remove leading/trailing spaces
)

gilts_info = gilts_info[
    gilts_info["ISIN Code"].notna() &
    (gilts_info["ISIN Code"].astype(str).str.strip() != "") &
    (gilts_info["ISIN Code"] != "ISIN Code")
]

dates = ["Redemption Date", "First Issue Date", "Current/Next Ex-dividend Date"]

gilts_info[dates] = gilts_info[dates].apply(pd.to_datetime)

# Show the cleaned DataFrame
gilts_info.head()

7,Conventional Gilts,ISIN Code,Redemption Date,First Issue Date,Dividend Dates,Current/Next Ex-dividend Date
1,1½% Treasury Gilt 2026,GB00BYZW3G56,2026-07-22,2016-02-18,22 Jan/Jul,2026-07-13
2,0 3/8% Treasury Gilt 2026,GB00BNNGP668,2026-10-22,2021-03-03,22 Apr/Oct,2026-04-13
3,4 1/8% Treasury Gilt 2027,GB00BL6C7720,2027-01-29,2022-10-13,29 Jan/Jul,2026-07-20
4,3¾% Treasury Gilt 2027,GB00BPSNB460,2027-03-07,2024-01-11,7 Mar/Sep,2026-08-26
5,1¼% Treasury Gilt 2027,GB00BDRHNP05,2027-07-22,2017-03-15,22 Jan/Jul,2026-07-13


### Define helper functions

In [ ]:
# -----------------------------
# Helper functions
# -----------------------------

def parse_date(date_str):
    """
    Handles both formats:
    - '2026-07-22'
    - '22/10/2026'
    """
    date_str = str(date_str).strip()
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d %b %Y"):
        try:
            return datetime.strptime(date_str, fmt)
        except:
            continue
    raise ValueError(f"Unknown date format: {date_str}")

def accrued_interest(coupon_rate, settlement_date, last_coupon_date, face_value=100):
    """
    Calculate accrued interest to add to clean price.
    """
    days_between_coupons = (settlement_date - last_coupon_date).days
    coupon_period_days = 182.5  # approx 6 months
    semi_coupon = coupon_rate / 2 * face_value
    return semi_coupon * (days_between_coupons / coupon_period_days)

def generate_coupon_dates(first_issue, redemption, dividend_dates):
    """
    Generate all coupon dates between first issue and redemption.
    dividend_dates: string like '22 Jan/Jul'
    """
    start_year = first_issue.year
    end_year = redemption.year
    schedule = []

    day_months = []
    for dm in dividend_dates.split('/'):
        day, mon = dm.split()
        day_months.append((int(day), mon))

    for y in range(start_year, end_year + 1):
        for day, mon in day_months:
            try:
                dt = datetime.strptime(f"{day} {mon} {y}", "%d %b %Y")
            except:
                continue
            if dt > first_issue and dt <= redemption:
                schedule.append(dt)
    return sorted(schedule)

def simplified_yield(dirty_price, coupons, face_value, years_to_maturity):
    """
    Compute after-tax annual yield assuming all cashflows are at maturity
    """
    total_inflows = sum(coupons) + face_value
    r = (total_inflows / dirty_price) ** (1 / years_to_maturity) - 1
    return r

### Example usage

In [ ]:
# -----------------------------
# Example input tables
# -----------------------------

# -----------------------------
# Compute simplified after-tax yield for each gilt
# -----------------------------
settlement_date = datetime.today()

results = []

for _, row in market_data.iterrows():
    clean_price = row["Price"]
    coupon_rate = row["Coupon (%)"]
    maturity = parse_date(row["Maturity"])

    # Match gilt details
    details = details_data[details_data["Conventional Gilts"] == row["Issuer"]].iloc[0]
    first_issue = parse_date(details["First Issue Date"])
    dividend_dates = details["Dividend Dates"]

    # Coupon schedule
    coupon_schedule = generate_coupon_dates(first_issue, maturity, dividend_dates)
    last_coupon = max([d for d in coupon_schedule if d <= settlement])

    # Dirty price = clean + accrued interest
    dirty_price = clean_price + accrued_interest(clean_price, coupon_rate, settlement, last_coupon)

    # Number of semi-annual coupons remaining (after settlement)
    remaining_coupons = [c for c in coupon_schedule if c > settlement]
    semi_coupon_value = coupon_rate / 2 * 100  # per £100 face
    coupons_after_tax = [semi_coupon_value * (1 - 0.45) for _ in remaining_coupons]

    # Fractional years to maturity
    years_to_maturity = (maturity - settlement).days / 365

    # Simplified annual yield
    r = simplified_yield(dirty_price, coupons_after_tax, 100, years_to_maturity)

    results.append({
        "Gilt": row["Issuer"],
        "Dirty Price": round(dirty_price, 3),
        "After-tax annual yield": round(r * 100, 4)
    })

# -----------------------------
# Show results
# -----------------------------
results_df = pd.DataFrame(results)
print(results_df)

ValueError: not enough values to unpack (expected 2, got 1)